# DB-QSP
https://arxiv.org/pdf/2504.01077

In [115]:
from qrisp import *
from qrisp.operators import X, Y, Z
from qrisp.jasp import q_fori_loop, q_cond, check_for_tracing_mode
from jax import lax
import scipy as sp
import numpy as np
import jax.numpy as jnp
import pytest

## Example model: XXZ

In [116]:
from qrisp.vqe.problems.heisenberg import create_heisenberg_init_function, heisenberg_problem, create_heisenberg_hamiltonian
L = 5
G = nx.Graph()
G.add_edges_from([(k,(k+1)%L) for k in range(L-1)]) 
J = 1
B = 0.5
H = create_heisenberg_hamiltonian(G, J, B)
print(H)

X(0)*X(1) + X(1)*X(2) + X(2)*X(3) + X(3)*X(4) + Y(0)*Y(1) + Y(1)*Y(2) + Y(2)*Y(3) + Y(3)*Y(4) + 0.5*Z(0) + Z(0)*Z(1) + 0.5*Z(1) + Z(1)*Z(2) + 0.5*Z(2) + Z(2)*Z(3) + 0.5*Z(3) + Z(3)*Z(4) + 0.5*Z(4)


In [117]:
# Define scaling factor
F = 1

def exp_H(qv, t):
    H.trotterization(method='commuting')(qv,t/F,5)

# Hamiltonian simulation via second order Suzuki-Trotter formula with 2 steps
def exp_H_2(qv, t):
    H.trotterization(order=2,method='commuting')(qv,t/F,2)

# Calculate E and V

In [232]:
# in qrisp
def calculate_EV(H, state_prep):
    H_2 = H**2
    E = H.expectation_value(state_prep, diagonalisation_method="commuting")()
    E_2 = H_2.expectation_value(state_prep, diagonalisation_method="commuting")()
    
    V = E_2 - E**2
    
    return E, V

# matrix, for tests only
def compute_moments(psi, H):
    psi = np.array([psi]).transpose()
    E = (psi.conj().T @ H @ psi)[0,0].real
    S = (psi.conj().T @ H @ H @ psi)[0,0].real
    return E, S, S - E**2

## calculate s and phase
$$
\frac{(H-zI)\ket{\Psi}}{\|(H-zI)\ket\Psi\|}   =e^{i\theta\Psi}e^{s_{\Psi}[\Psi, H]}\ket\Psi.
$$

with $s_k = \frac{-1}{\sqrt{V_k}}\arccos\left(\frac{|E_{k}-z_{k}|}{\sqrt{V_{k}+|E_{k}-z_{k}|^2}}\right)$
and $\theta_k = \arg\left(\frac{E_k-z_k}{|E_k-z_k|}\right).$

In [121]:
def QSP_unitary_synthesis_params(E, V, z):
    diff = E - z
    s = -1/jnp.sqrt(V)*jnp.arccos(jnp.abs(diff)/jnp.sqrt(V+jnp.abs(diff)**2))
    theta = jnp.angle(diff)
    return s, theta

## DB-QSP steps
$$
\frac{(H-zI)\ket{\Psi}}{\|(H-zI)\ket\Psi\|}   =e^{i\theta\Psi}e^{s_{\Psi}[\Psi, H]}\ket\Psi.
$$
$$
e^{s_\Psi[\Psi,H]} = \left(
e^{is_\Psi^{(N)} \Psi}e^{is_\Psi^{(N)} H}
e^{-is_\Psi^{(N)} \Psi}e^{-is_\Psi^{(N)} H}
\right)^N \nonumber+O(s_\Psi^{3/2}/\sqrt N)\ , 
$$


### 1 step

In [ ]:
def DB_QSP(qarg, U0, H, exp_H, z, N):
    U0(qarg)
    def conjugator(qarg):
        with invert():
            # here U0 needs to gnerate Psi
            U0(qarg)
            
    def reflection(qarg, t_):
        with conjugate(conjugator)(qarg):
            if isinstance(qarg,QuantumArray):
                qubits = sum([qv.reg for qv in qarg.flatten()], [])
                mcp(t_, qubits, ctrl_state=0, method="khattar")
            else:
                mcp(t_, qarg, ctrl_state=0, method="khattar")
    E, V = calculate_EV(H, qarg)
    s, theta = QSP_unitary_synthesis_params(E, V, z)
    s_ = jnp.sqrt(jnp.abs(s)/N)
    for _ in range(N):
        exp_H(qarg, s_)
        reflection(qarg, s_)
        exp_H(qv, -s_)
        reflection(qarg, -s_)
    reflection(qarg, -theta)
    return s, theta

In [235]:
# Apply 1 step of DB-QSP  w.r.t. |Psi_0> = U0|0>
def DB_QSP_1s(qarg, U0, H, exp_H, z, N, s, theta):
    
    def conjugator(qarg):
        with invert():
            # here U0 needs to gnerate Psi
            U0(qarg)
            
    def reflection(qarg, t_):
        with conjugate(conjugator)(qarg):
            if isinstance(qarg,QuantumArray):
                qubits = sum([qv.reg for qv in qarg.flatten()], [])
                mcp(t_, qubits, ctrl_state=0, method="khattar")
            else:
                mcp(t_, qarg, ctrl_state=0, method="khattar")

    U0(qarg)

    s_ = jnp.sqrt(jnp.abs(s)/N)
    for _ in range(N):
        exp_H(qarg, s_)
        reflection(qarg, -s_)
        exp_H(qarg, -s_)
        reflection(qarg, s_)

    reflection(qarg, theta)

### Test group commutator formula

In [228]:
psi = np.zeros(2**L)
psi[2**(L-2)] = 1
psi = psi/np.linalg.norm(psi)
H_matrix = H.to_array()
psi_dm = np.outer(psi, psi.conj())
s = -0.2

comm = psi_dm@H_matrix - H_matrix@psi_dm
U_exact = sp.linalg.expm(s*comm)
# build the GC approx once
N = 5
for i in range(1, N+1):
    a = np.sqrt(abs(s/i))
    P, Hm = psi_dm, H_matrix
    A, B = 1j*a*P, 1j*a*Hm
    U_gc = np.eye(2**L, 2**L)
    for _ in range(i):
        U_gc = sp.linalg.expm(A) @ sp.linalg.expm(B) @ sp.linalg.expm(-A) @ sp.linalg.expm(-B) @ U_gc

    # Compare them:

    print(f"N={i} ‖U_exact - U_gc‖₂ =", np.linalg.norm(U_exact - U_gc))

N=1 ‖U_exact - U_gc‖₂ = 0.7996376392834138
N=2 ‖U_exact - U_gc‖₂ = 0.6324286670868754
N=3 ‖U_exact - U_gc‖₂ = 0.5365690210112508
N=4 ‖U_exact - U_gc‖₂ = 0.47382639100210594
N=5 ‖U_exact - U_gc‖₂ = 0.4288378406008772


### Numerical checks of DB-QSP

In [237]:
# example numpy calculation
psi = np.zeros(2**L)
psi[2**(L-2)] = 1
psi = psi/np.linalg.norm(psi)
H_matrix = H.to_array()
psi_dm = np.outer(psi, psi.conj())

zk = -0.2

# target state
I = np.eye(2**L, 2**L)
psi_target = (H_matrix-zk*I)@ psi
psi_target /= np.linalg.norm(psi_target)

# db-qsp state
E, _, V = compute_moments(psi, H_matrix)
print("Initial EV", (E, V))
s, theta = QSP_unitary_synthesis_params(E, V, zk)
print("     s, theta", (s, theta))
psi_qsp = sp.linalg.expm(1j*theta*psi_dm) @ sp.linalg.expm(s*(psi_dm@H_matrix-H_matrix@psi_dm)) @ psi
N = 5
s_ = np.sqrt(np.abs(s/N))
U_qsp_gc = np.eye(2**L, 2**L)
for _ in range(N):
    U_qsp_gc = sp.linalg.expm(1j*theta*psi_dm) @ sp.linalg.expm(1j*s_*psi_dm) @ sp.linalg.expm(1j*s_*H_matrix) @ sp.linalg.expm(-1j*s_*psi_dm) @ sp.linalg.expm(-1j*s_*H_matrix) @ U_qsp_gc
psi_qsp_gc = U_qsp_gc @ psi
print("     Fidelity", abs(np.vdot(psi_target, psi_qsp))**2)
print("     Fidelity_GC", abs(np.vdot(psi_target, psi_qsp_gc))**2)
E_qsp = np.vdot(psi_qsp, H_matrix @ psi_qsp).real
V_qsp = np.vdot(psi_qsp, H_matrix @ H_matrix @ psi_qsp).real - E_qsp**2
print("After DB-QSP", (E_qsp, V_qsp))
E_qsp_gc = np.vdot(psi_qsp_gc, H_matrix @ psi_qsp_gc).real
V_qsp_gc = np.vdot(psi_qsp_gc, H_matrix @ H_matrix @ psi_qsp_gc).real - E_qsp_gc**2
print("After DB-QSP_GC", (E_qsp_gc, V_qsp_gc))

Initial EV (np.float64(1.5), np.float64(8.0))
     s, theta (Array(-0.36402278, dtype=float64), Array(0., dtype=float64))
     Fidelity 0.9999999999999998
     Fidelity_GC 0.7226945770143561
After DB-QSP (np.float64(4.732323232323233), np.float64(2.988266503418018))
After DB-QSP_GC (np.float64(3.8385758654375617), np.float64(6.538545485626658))


### Check DB-QSP circuit

In [239]:
def U0(qv):
    x(qv[1])

def state_prep():
    qarg = QuantumVariable(L)
    U0(qarg) # Prepares the initial state |psi>
    return qarg

z = -0.2
N = 5
E, V = calculate_EV(H, state_prep)
s, theta = QSP_unitary_synthesis_params(E, V, z)
print("     s, theta", (s, theta))
print("Initial EV", E, V)


def state_prep_1():
    qv = QuantumVariable(L)
    DB_QSP_1s(qv, U0, H, exp_H_2, z, N, s, theta)
    return qv


E1, V1 = calculate_EV(H, state_prep_1)
print("After DB-QSP", E1, V1)

     s, theta (Array(-0.36397321, dtype=float64, weak_type=True), Array(0., dtype=float64))
Initial EV 1.5011251283349505 7.995863307529726
After DB-QSP 3.857520327250681 6.427788634514709                                     


## multiple steps

$$
\frac{p(H)\ket{\Psi_0}}{\|p(H)\ket{\Psi_0}\|}=\prod_{k=0}^{K-1} e^{i \theta_k \Psi_{k}}  e^{s_{k}[\Psi_{k},H]}\ket{\Psi_0},
$$

In [253]:
# Apply multiple steps of DB-QSP 
def DB_QSP(U0, H, exp_H, z_list, N=2):

    L = H.find_minimal_qubit_amount()
    U_list = [U0]

    for k, zk in enumerate(z_list):

        def state_prep():
            qv = QuantumVariable(L)
            U_list[k](qv)
            return qv

        E, V = calculate_EV(H, state_prep)
        s, theta = QSP_unitary_synthesis_params(E, V, zk)

        def U_new(qarg):
            return DB_QSP_1s(qarg, U_list[k], H, exp_H, zk, N, s, theta)
        
        U_list.append(U_new)
    return U_list

In [270]:
z_list = [-0.2]
pH = [I]
for k, zk in enumerate(z_list):
    pH.append((H_matrix - zk * I)@pH[-1])
energy_expect = lambda U: (np.vdot(U@psi, H_matrix@U@psi)/np.linalg.norm(U@psi)**2).real
energies = [energy_expect(U) for U in pH]
print(energies)

[np.float64(1.5), np.float64(4.732323232323232)]


In [268]:
U_list = DB_QSP(U0, H, exp_H_2, z_list, N=5)
for U in U_list:
    def state_prep():
        qv = QuantumVariable(L)
        U(qv)
        return qv
    print(calculate_EV(H, state_prep))

(1.503403513213226, 7.973189360739465)                                               
(3.849075362720552, 6.508043464349308)                                               


### Multiple steps

In [271]:
z_list = [-0.2, 0.1]
pH = [I]
for k, zk in enumerate(z_list):
    pH.append((H_matrix - zk * I)@pH[-1])
energy_expect = lambda U: (np.vdot(U@psi, H_matrix@U@psi)/np.linalg.norm(U@psi)**2).real
energies = [energy_expect(U) for U in pH]
print(energies)

[np.float64(1.5), np.float64(4.732323232323232), np.float64(5.211936246264428)]


In [272]:
U_list = DB_QSP(U0, H, exp_H_2, z_list, N=5)
for U in U_list:
    def state_prep():
        qv = QuantumVariable(L)
        U(qv)
        return qv
    print(calculate_EV(H, state_prep))

RecursionError: maximum recursion depth exceeded